`continuous_carry` is deliberately assigned to CPU because it is one long sequential chain. Run its five two-fold shards on Colab one at a time; rerunning the same cell resumes checkpoints stored on Drive.

In [ ]:
import json, subprocess, sys
POLICY_PLAN = LOCAL/'v05_policy_job_plan.json'
policy_plan = json.loads(POLICY_PLAN.read_text())
cpu_jobs = [j['job_id'] for j in policy_plan['jobs'] if j['execution_target']=='cpu']
print('CPU jobs:', *cpu_jobs, sep='\n- ')
CPU_JOB_ID = cpu_jobs[0]  # Change to the next ID after this shard completes.
os.chdir(DRIVE_ROOT)
subprocess.run([sys.executable, str(LOCAL/'run_planned_job.py'), '--plan', str(POLICY_PLAN), '--job', CPU_JOB_ID, '--data', str(LOCAL/'data/ohlc_export.csv')], check=True)

# ARSH v0.5 revised — CPU coordinator
Colab **CPU** only: verify bundle, merge CUDA shards, finalize outputs, bootstrap and report. It never refits a missing CUDA fold.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, shutil, zipfile
DRIVE_ROOT = Path('/content/drive/MyDrive/ARSH_Revised')
BUNDLE = DRIVE_ROOT / 'ARSH_v0.5_CUDA_worker.zip'
LOCAL = Path('/content/ARSH_v05_revised')
assert BUNDLE.is_file(), BUNDLE
if LOCAL.exists(): shutil.rmtree(LOCAL)
with zipfile.ZipFile(BUNDLE) as z: z.extractall(LOCAL)
os.chdir(LOCAL)
print('Project:', LOCAL)

In [ ]:
!python verify_cuda_bundle.py
!python -m pip install -q -r requirements.txt
!python -m unittest -v test_arsh_v05.py

Copy completed CUDA `runs_v05_revised` folders into `MyDrive/ARSH_Revised/`. Then finalize a plan below.

In [ ]:
# Work directly under Drive so relative paths in the immutable plan resolve correctly.
os.chdir(DRIVE_ROOT)
PLAN = LOCAL / 'v05_policy_job_plan.json'
!python {LOCAL/'finalize_plan.py'} --plan {PLAN} --data {LOCAL/'data/ohlc_export.csv'}
print('Policy experiments finalized.')

In [ ]:
!python {LOCAL/'select_policy.py'} \
  runs_v05_revised/policy_continuous/merged \
  runs_v05_revised/policy_daily/merged \
  runs_v05_revised/policy_session/merged \
  --output runs_v05_revised/policy_decision

Generate the post-policy plan only after validation chooses the reset policy. Download this JSON to the extracted CUDA bundle, run every listed job, and copy the completed folders back to Drive.

In [ ]:
import json, subprocess, sys
decision = json.loads((DRIVE_ROOT/'runs_v05_revised/policy_decision/policy_decision.json').read_text())
POST_PLAN = DRIVE_ROOT / 'v05_post_policy_job_plan.json'
subprocess.run([sys.executable, str(LOCAL/'create_job_plan.py'), '--data', str(LOCAL/'data/ohlc_export.csv'), '--phase', 'post_policy', '--policy', decision['preferred_policy'], '--output', str(POST_PLAN), '--runs-root', 'runs_v05_revised'], cwd=DRIVE_ROOT, check=True)
post = json.loads(POST_PLAN.read_text())
print('Plan:', POST_PLAN)
print('CPU jobs:', sum(j['execution_target']=='cpu' for j in post['jobs']), 'CUDA jobs:', sum(j['execution_target']=='cuda' for j in post['jobs']))

In [ ]:
# Run after all post-policy CUDA shard folders have been copied back to Drive.
subprocess.run([sys.executable, str(LOCAL/'finalize_plan.py'), '--plan', str(POST_PLAN), '--data', str(LOCAL/'data/ohlc_export.csv')], cwd=DRIVE_ROOT, check=True)
print('Post-policy experiments finalized.')

In [ ]:
# CPU-only distribution and timestamp diagnostics.
DIST = DRIVE_ROOT/'runs_v05_revised/distribution_diagnostics'
TIME = DRIVE_ROOT/'runs_v05_revised/timestamp_regime'
MAIN = DRIVE_ROOT/'runs_v05_revised/main_revised/merged'
subprocess.run([sys.executable, str(LOCAL/'distribution_diagnostics.py'), '--data', str(LOCAL/'data/ohlc_export.csv'), '--output', str(DIST)], check=True)
subprocess.run([sys.executable, str(LOCAL/'timestamp_regime_analysis.py'), '--data', str(LOCAL/'data/ohlc_export.csv'), '--main-output', str(MAIN), '--output', str(TIME)], check=True)

The selected 500-iteration convergence audit remains a CUDA task. Copy the finalized `main_revised/merged` folder to the CUDA machine, run fold shards with `audit_convergence_500.py`, merge them with `merge_convergence_audits.py`, and copy the merged audit to `runs_v05_revised/convergence500_merged` on Drive.

In [ ]:
# Final gate: refuses to approve v0.6 if any required evidence is absent.
plan_policy = json.loads((LOCAL/'v05_policy_job_plan.json').read_text())
plan_post = json.loads(POST_PLAN.read_text())
names = [x['id'] for x in plan_policy['experiments'] + plan_post['experiments']]
folders = [DRIVE_ROOT/'runs_v05_revised'/name/'merged' for name in names]
CONV = DRIVE_ROOT/'runs_v05_revised/convergence500_merged'
command = [sys.executable, str(LOCAL/'compare_sensitivities.py'), *map(str, folders), '--output', str(DRIVE_ROOT/'runs_v05_revised/v05_completion'), '--convergence-output', str(CONV), '--distribution-output', str(DIST), '--timestamp-output', str(TIME), '--require-complete']
subprocess.run(command, check=True)